In [ ]:
# %pip install -r requirements.txt

     ---------------------------------------- 0.0/80.3 kB ? eta -:--:--
     ---------------------------------------- 0.0/80.3 kB ? eta -:--:--
     ----- ---------------------------------- 10.2/80.3 kB ? eta -:--:--
     -------------- ----------------------- 30.7/80.3 kB 325.1 kB/s eta 0:00:01
     ----------------------------- -------- 61.4/80.3 kB 469.7 kB/s eta 0:00:01
     -------------------------------------- 80.3/80.3 kB 496.6 kB/s eta 0:00:00
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
     ---------------------------------------- 0.0/121.0 kB ? eta -:--:--
     -------------------------------------- 121.0/121.0 kB 3.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.6/9.9 MB 20.5 MB/s eta 0:00:01
   --- ----------------------------------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\mattm\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Bloco 1:
Todas as importações e as funções de tratamento de texto.

In [4]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
import unicodedata
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def normalizar_texto(valor):
    if pd.isna(valor): return valor
    valor = str(valor).strip().lower()
    valor = unicodedata.normalize("NFKD", valor)
    return "".join([c for c in valor if not unicodedata.combining(c)])

def converter_sim_nao(valor):
    if pd.isna(valor): return valor
    valor_norm = normalizar_texto(valor)
    mapa = {"sim": 1, "s": 1, "yes": 1, "y": 1, "nao": 0, "não": 0, "n": 0, "no": 0}
    return mapa.get(valor_norm, valor)

## Bloco 2:
Configuração de pastas, leitura do CSV e o Filtro Anticola (Vazamento).

In [5]:
PASTA_PROJETO = os.getcwd() # Ajustado para funcionar direto no Jupyter
ARQUIVO_BASE_ORIGINAL = os.path.join(PASTA_PROJETO, "vigitel.csv")
ARQUIVO_VARIAVEIS = os.path.join(PASTA_PROJETO, "variaveis.txt")
ARQUIVO_BASE_TRATADA = os.path.join(PASTA_PROJETO, "base_tratada_sem_vazamento.csv")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PASTA_RESULTADOS = os.path.join(PASTA_PROJETO, "resultados_sem_vazamento", RUN_ID)
PASTA_MODELOS = os.path.join(PASTA_RESULTADOS, "modelos_salvos")
os.makedirs(PASTA_RESULTADOS, exist_ok=True)
os.makedirs(PASTA_MODELOS, exist_ok=True)

TARGET = "hart"
RANDOM_STATE = 42
N_FOLDS_AVALIACAO = 5
N_FOLDS_TUNING = 3
LIMITE_ALERTA_CORRELACAO = 0.30

lista_variaveis = pd.read_csv(ARQUIVO_VARIAVEIS, header=None)[0].astype(str).str.strip().tolist()
df = pd.read_csv(ARQUIVO_BASE_ORIGINAL, encoding="latin1", sep=",", on_bad_lines="skip") # Simplificado para o notebook

variaveis_validas = [col for col in lista_variaveis if col in df.columns]
df = df[variaveis_validas]

colunas_texto = df.select_dtypes(include=["object", "string"]).columns
df[colunas_texto] = df[colunas_texto].apply(lambda col: col.map(converter_sim_nao))
df = df.dropna(subset=[TARGET])

y = df[TARGET]
X = df.drop([TARGET], axis=1)

termos_vazamento = ["has", "hipert", "hipertens", "pressao", "pressão"]
variaveis_remover_vazamento = [col for col in X.columns if any(t in col.lower() for t in termos_vazamento)]
X = X.drop(columns=variaveis_remover_vazamento)

print(f"Base carregada. Linhas: {X.shape[0]} | Colunas restantes: {X.shape[1]}")

C:\Users\mattm\AppData\Local\Temp\ipykernel_7704\915171651.py:19: DtypeWarning: Columns (0: q10, 1: q12, 2: q14, 3: q14a, 4: q15, 5: q21a, 6: q22, 7: q23a, 8: q24, 9: q30, 10: q31, 11: q32a, 12: q33, 13: q35, 14: q37a, 15: q37b, 16: q38a, 17: q38b, 18: q41, 19: q43, 20: q43a, 21: q55a, 22: q57, 23: q58, 24: q59, 25: q63, 26: q73, 27: q76, 28: q77, 29: q78, 30: q85, 31: q86, 32: q87, 33: r101, 34: r102, 35: r103, 36: r104, 37: r105, 38: flvreg, 39: flvreco, 40: gordura, 41: leiteint, 42: refri5, 43: atilaz, 44: inativo, 45: regiao, 46: refritl5, 47: feijao5, 48: ati_l, 49: ativo_livre, 50: ina_livre, 51: inat_des_trab, 52: inat_des_esc, 53: inat_desl, 54: atilaz_trans, 55: q16, 56: q21, 57: q23, 58: q32, 59: q34, 60: q34a, 61: q34b, 62: q34c, 63: q34d, 64: q39, 65: q40, 66: q40a, 67: q72, 68: q79, 69: q80, 70: q81, 71: q82, 72: q83, 73: q83a, 74: r107, 75: r108, 76: iddmamo, 77: mamo, 78: mamodois, 79: papa, 80: papatres, 81: q25, 82: q26, 83: q66a, 84: r109, 85: r110, 86: r111, 87: r11

Base carregada. Linhas: 833217 | Colunas restantes: 90


## Bloco 3:
Montagem da esteira de automação (ColumnTransformer).

In [6]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
    ]
)

## Bloco 4:
Define as listas de tentativas de parâmetros (GridSearchCV), roda as rodadas de testes (Validação Cruzada), avalia o Recall, salva os modelos físicos e desenha os gráficos automaticamente na pasta do projeto.

In [7]:
# ============================================================
# MODELOS-BASE E GRADES DE HIPERPARÂMETROS PARA TUNING
# ============================================================
# Definimos os dicionários contendo os modelos e as listas de botões
# de regulagem que o GridSearchCV vai testar de forma combinada.

modelos_base = {
    "Regressao_Logistica": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random_Forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=1,
        class_weight="balanced"
    ),
    "Arvore_Decisao": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

grades_hiperparametros = {
    "Regressao_Logistica": {
        "modelo__C": [0.01, 0.1, 1, 10]
    },
    "Random_Forest": {
        "modelo__n_estimators": [200, 500],
        "modelo__max_depth": [8, 12],
        "modelo__min_samples_leaf": [5, 10]
    },
    "Arvore_Decisao": {
        "modelo__max_depth": [6, 10, 14],
        "modelo__min_samples_leaf": [5, 10, 20]
    }
}

# Configura as divisões para as rodadas de validação cruzada
cv_tuning = StratifiedKFold(n_splits=N_FOLDS_TUNING, shuffle=True, random_state=RANDOM_STATE)
cv_avaliacao = StratifiedKFold(n_splits=N_FOLDS_AVALIACAO, shuffle=True, random_state=RANDOM_STATE)

metricas_cv = ["accuracy", "precision", "recall", "f1", "roc_auc"]

# Listas auxiliares para armazenar os relatórios
resultados = []
resultados_cv = []
resultados_cv_folds = {}
melhores_parametros = {}
modelos_treinados = {}
curvas_roc = {}
curvas_pr = {}

# ============================================================
# EXECUÇÃO DO LAÇO DE TREINAMENTO E VALIDAÇÃO
# ============================================================
for nome_modelo, algoritmo_base in modelos_base.items():
    print("\n" + "=" * 80)
    print(f"Processando o Modelo: {nome_modelo}")
    print("=" * 80)

    # Conecta a esteira automática ao algoritmo atual da rodada
    pipeline_base = Pipeline([
        ("prep", preprocessor),
        ("modelo", algoritmo_base)
    ])

    # 1) Executa o Tuning de Hiperparâmetros no conjunto de treino
    print(f"Buscando melhores hiperparâmetros ({N_FOLDS_TUNING} folds)...")
    grid = GridSearchCV(
        estimator=pipeline_base,
        param_grid=grades_hiperparametros[nome_modelo],
        scoring="roc_auc",
        cv=cv_tuning,
        n_jobs=-1,  # Usa todos os núcleos do processador para acelerar localmente
        refit=True
    )
    grid.fit(X_treino, y_treino)

    pipeline = grid.best_estimator_
    melhores_parametros[nome_modelo] = grid.best_params_

    print(f"Melhores parâmetros encontrados: {grid.best_params_}")
    print(f"Melhor AUC-ROC durante o tuning: {grid.best_score_:.4f}")

    # 2) Executa a Validação Cruzada de 5 Folds para conferência estável
    print(f"Executando validação cruzada de avaliação ({N_FOLDS_AVALIACAO} folds)...")
    cv_resultado = cross_validate(
        pipeline, X, y, cv=cv_avaliacao, scoring=metricas_cv, n_jobs=-1
    )

    linha_cv = {"modelo": nome_modelo}
    for metrica in metricas_cv:
        chave = f"test_{metrica}"
        linha_cv[f"{metrica}_media"] = cv_resultado[chave].mean()
        linha_cv[f"{metrica}_desvio"] = cv_resultado[chave].std()

    resultados_cv.append(linha_cv)
    resultados_cv_folds[nome_modelo] = cv_resultado

    # 3) Faz a avaliação final usando o split isolado de Teste
    pred = pipeline.predict(X_teste)
    proba = pipeline.predict_proba(X_teste)[:, 1]

    acc = accuracy_score(y_teste, pred)
    precision = precision_score(y_teste, pred, zero_division=0)
    recall = recall_score(y_teste, pred, zero_division=0)
    f1 = f1_score(y_teste, pred, zero_division=0)
    auc = roc_auc_score(y_teste, proba)

    resultados.append({
        "modelo": nome_modelo, "accuracy": acc, "precision": precision,
        "recall": recall, "f1_score": f1, "roc_auc": auc
    })

    modelos_treinados[nome_modelo] = pipeline

    print("\nClassification Report (split treino/teste):")
    print(classification_report(y_teste, pred, zero_division=0))

    # Coleta as taxas para plotar as curvas comparativas mais tarde
    fpr, tpr, _ = roc_curve(y_teste, proba)
    curvas_roc[nome_modelo] = (fpr, tpr, auc)
    precisao_pr, recall_pr, _ = precision_recall_curve(y_teste, proba)
    ap = average_precision_score(y_teste, proba)
    curvas_pr[nome_modelo] = (precisao_pr, recall_pr, ap)

    # 4) Desenha e salva a Matriz de Confusão com termos amigáveis
    ConfusionMatrixDisplay.from_predictions(
        y_teste, pred, display_labels=["Sem Hipertensão", "Com Hipertensão"],
        values_format="d", cmap="Blues"
    )
    plt.title(f"Matriz de Confusão - {nome_modelo}")
    plt.tight_layout()
    caminho_matriz = os.path.join(PASTA_RESULTADOS, f"matriz_confusao_{nome_modelo}.png")
    plt.savefig(caminho_matriz, dpi=300)
    plt.close()

    # 5) Exporta o arquivo físico do modelo pronto para reuso (.joblib)
    caminho_modelo_salvo = os.path.join(PASTA_MODELOS, f"{nome_modelo}.joblib")
    joblib.dump(pipeline, caminho_modelo_salvo)

# ============================================================
# EXPORTAÇÃO DE RELATÓRIOS EM TEXTO E CSV
# ============================================================
caminho_hiperparametros = os.path.join(PASTA_RESULTADOS, "melhores_hiperparametros.txt")
with open(caminho_hiperparametros, "w", encoding="utf-8") as arquivo:
    for nome_modelo, params in melhores_parametros.items():
        arquivo.write(f"{nome_modelo}: {params}\n")

resultados_cv_df = pd.DataFrame(resultados_cv).sort_values(by="roc_auc_media", ascending=False)
resultados_cv_df.to_csv(os.path.join(PASTA_RESULTADOS, "comparacao_modelos_cross_validation_sem_vazamento.csv"), index=False, encoding="utf-8-sig")

resultados_df = pd.DataFrame(resultados).sort_values(by="roc_auc", ascending=False)
resultados_df.to_csv(os.path.join(PASTA_RESULTADOS, "comparacao_modelos_sem_vazamento.csv"), index=False, encoding="utf-8-sig")

# ============================================================
# DESENHO DOS GRÁFICOS VISUAIS PARA O PDF FINAL
# ============================================================
# Gráfico de Linhas: Comparativo de Métricas Gerais
plt.figure(figsize=(10, 6))
x_axis = resultados_df["modelo"]
plt.plot(x_axis, resultados_df["accuracy"], marker="o", label="Accuracy")
plt.plot(x_axis, resultados_df["precision"], marker="o", label="Precision")
plt.plot(x_axis, resultados_df["recall"], marker="o", label="Recall")
plt.plot(x_axis, resultados_df["f1_score"], marker="o", label="F1-score")
plt.plot(x_axis, resultados_df["roc_auc"], marker="o", label="AUC-ROC")
plt.title("Comparação de Métricas dos Modelos (split treino/teste)")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_RESULTADOS, "comparacao_metricas_modelos_sem_vazamento.png"), dpi=300)
plt.close()

# Gráfico de Linhas: Curva ROC Comparativa
plt.figure(figsize=(8, 8))
for nome_modelo, (fpr, tpr, auc_modelo) in curvas_roc.items():
    plt.plot(fpr, tpr, label=f"{nome_modelo} (AUC = {auc_modelo:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Modelo aleatório")
plt.xlabel("Taxa de Falsos Positivos")
plt.ylabel("Taxa de Verdadeiros Positivos (Recall)")
plt.title("Curva ROC - Comparação entre Modelos")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PASTA_RESULTADOS, "curva_roc_comparativa.png"), dpi=300)
plt.close()

# Extração de Feature Importance (Random Forest)
modelo_rf = modelos_treinados["Random_Forest"]
feature_names = modelo_rf.named_steps["prep"].get_feature_names_out()
importance = modelo_rf.named_steps["modelo"].feature_importances_

importance_df = pd.DataFrame({"variavel": feature_names, "importancia": importance}).sort_values("importancia", ascending=False)
importance_df.to_csv(os.path.join(PASTA_RESULTADOS, "feature_importance_random_forest_sem_vazamento.csv"), index=False, encoding="utf-8-sig")

# Gráfico de Barras: Top 20 variáveis explicativas do modelo campeão
top20 = importance_df.head(20)
plt.figure(figsize=(10, 8))
plt.barh(top20["variavel"], top20["importancia"])
plt.gca().invert_yaxis()
plt.title("Top 20 Variáveis Mais Importantes - Random Forest")
plt.tight_layout()
plt.savefig(os.path.join(PASTA_RESULTADOS, "top20_feature_importance_random_forest_sem_vazamento.png"), dpi=300)
plt.close()

# Imprime o grande veredito na tela do Jupyter
print("\n" + "=" * 80)
print("DIAGNÓSTICO DO PIPELINE CONCLUÍDO COM SUCESSO")
print("=" * 80)
melhor_modelo_cv = resultados_cv_df.iloc[0]
print(f"Modelo Vencedor (Validação Cruzada): {melhor_modelo_cv['modelo']}")
print(f"AUC-ROC Médio obtido: {melhor_modelo_cv['roc_auc_media']:.4f} ± {melhor_modelo_cv['roc_auc_desvio']:.4f}")
print(f"Todos os artefatos visuais e relatórios foram salvos na pasta: {PASTA_RESULTADOS}")


Processando o Modelo: Regressao_Logistica
Buscando melhores hiperparâmetros (3 folds)...
Melhores parâmetros encontrados: {'modelo__C': 0.1}
Melhor AUC-ROC durante o tuning: 0.7727
Executando validação cruzada de avaliação (5 folds)...

Classification Report (split treino/teste):
              precision    recall  f1-score   support

           0       0.84      0.72      0.77    115605
           1       0.52      0.69      0.59     51039

    accuracy                           0.71    166644
   macro avg       0.68      0.70      0.68    166644
weighted avg       0.74      0.71      0.72    166644


Processando o Modelo: Random_Forest
Buscando melhores hiperparâmetros (3 folds)...
Melhores parâmetros encontrados: {'modelo__max_depth': 12, 'modelo__min_samples_leaf': 5, 'modelo__n_estimators': 200}
Melhor AUC-ROC durante o tuning: 0.7735
Executando validação cruzada de avaliação (5 folds)...

Classification Report (split treino/teste):
              precision    recall  f1-score   su